# 🔬 OSCC Detection — Demo Notebook
**Microscope-Aided Deep Learning Framework for Early Oral Cancer Detection**

This notebook demonstrates the complete inference pipeline:
1. Load a test image
2. Extract patches
3. Extract CNN features
4. Run the full model (CNN → Attention → LSTM → MLP)
5. Visualise attention weights and Grad-CAM explanations

---

## 1. Setup and Imports

In [ ]:
import sys, os, glob, random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path

# Add src to path
PROJECT_ROOT = Path(os.getcwd()).parent
SRC_DIR = str(PROJECT_ROOT / 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Check GPU
gpus = tf.config.list_physical_devices('GPU')
print(f'TensorFlow {tf.__version__}')
print(f'GPU available: {len(gpus) > 0}')
if gpus:
    print(f'  GPUs: {gpus}')

# Import project modules
from patch_extractor import PatchExtractor
from cnn_feature_extractor import CNNFeatureExtractor
from mlp_classifier import get_risk_level, predict_with_risk
print('\n✅ All modules imported successfully.')

## 2. Load Test Images
Load one **Normal** and one **OSCC** image from the test set.

In [ ]:
import cv2

data_dir = PROJECT_ROOT / 'data' / 'test'

def load_sample_image(cls_name):
    cls_dir = data_dir / cls_name
    if not cls_dir.exists():
        print(f'⚠️  {cls_dir} not found')
        return None, None
    files = sorted([f for f in cls_dir.iterdir() if f.suffix.lower() in {'.jpg','.jpeg','.png'}])
    if not files:
        print(f'⚠️  No images in {cls_dir}')
        return None, None
    path = files[0]
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img, str(path)

normal_img, normal_path = load_sample_image('normal')
oscc_img, oscc_path = load_sample_image('oscc')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, img, label in zip(axes, [normal_img, oscc_img], ['Normal', 'OSCC']):
    if img is not None:
        ax.imshow(img)
        ax.set_title(f'{label}  ({img.shape[1]}×{img.shape[0]})', fontsize=13, fontweight='bold')
        ax.axis('off')
        print(f'{label}: shape={img.shape}, range=[{img.min()}, {img.max()}], mean={img.mean():.1f}')
plt.tight_layout()
plt.show()

## 3. Patch Extraction
Split each image into 224×224 non-overlapping patches.

In [ ]:
pe = PatchExtractor(patch_size=224, min_image_size=448, overlap=0)

for label, img_path in [('Normal', normal_path), ('OSCC', oscc_path)]:
    if img_path is None:
        continue
    patches, shape = pe.extract_from_file(img_path)
    print(f'\n{label}: {len(patches)} patches from {shape}')
    
    n_show = min(len(patches), 8)
    fig, axes = plt.subplots(1, n_show + 1, figsize=(3*(n_show+1), 3))
    
    orig = cv2.imread(img_path, cv2.IMREAD_COLOR)
    orig = cv2.cvtColor(orig, cv2.COLOR_BGR2RGB)
    axes[0].imshow(orig)
    axes[0].set_title('Original', fontsize=10, fontweight='bold')
    axes[0].axis('off')
    
    for j in range(n_show):
        axes[j+1].imshow(np.clip(patches[j], 0, 1))
        axes[j+1].set_title(f'Patch {j}', fontsize=9)
        axes[j+1].axis('off')
    
    plt.suptitle(f'{label} — Patch Extraction', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 4. Feature Extraction
Extract deep features from patches using pretrained EfficientNetB0.

In [ ]:
cnn = CNNFeatureExtractor(backbone='efficientnet', trainable=False)

if normal_path:
    patches, _ = pe.extract_from_file(normal_path)
    features = cnn.extract_features(patches[:4])
    print(f'Feature matrix shape: {features.shape}')
    print(f'Feature range: [{features.min():.4f}, {features.max():.4f}]')
    print(f'Feature mean:  {features.mean():.4f}')
    
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(features.flatten(), bins=50, color='#4ade80', alpha=0.8, edgecolor='white')
    ax.set_title('Feature Distribution (EfficientNetB0)', fontsize=13, fontweight='bold')
    ax.set_xlabel('Feature Value')
    ax.set_ylabel('Frequency')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 5. Model Prediction
Load the trained model and run the full pipeline.

In [ ]:
from model import build_oscc_model
import json

config_path = PROJECT_ROOT / 'outputs' / 'models' / 'model_config.json'
weights_path = PROJECT_ROOT / 'outputs' / 'models' / 'best_model.weights.h5'

# Load config if available
config = None
if config_path.exists():
    with open(config_path) as f:
        config = json.load(f)
    print(f'Config loaded: {config}')

model = build_oscc_model(config)

# Load weights if available
if weights_path.exists():
    model.load_weights(str(weights_path))
    print('✅ Trained weights loaded.')
else:
    alt = PROJECT_ROOT / 'outputs' / 'models' / 'final_model.weights.h5'
    if alt.exists():
        model.load_weights(str(alt))
        print('✅ Trained weights loaded (final).')
    else:
        print('⚠️  No trained weights found — predictions will be random.')

In [ ]:
# Run prediction on test images
num_patches = config.get('num_patches', 4) if config else 4

for label, img_path in [('Normal', normal_path), ('OSCC', oscc_path)]:
    if img_path is None:
        continue
    patches, _ = pe.extract_from_file(img_path)
    
    # Pad/truncate
    while len(patches) < num_patches:
        patches.append(patches[len(patches) % len(patches)])
    patches = patches[:num_patches]
    
    patch_array = np.stack(patches, axis=0)[np.newaxis, ...].astype(np.float32)
    output = model(patch_array, training=False)
    
    preds = output['predictions'].numpy()[0]
    attn = output['attention_weights'].numpy()[0].flatten()
    
    pred_cls = 'OSCC' if np.argmax(preds) == 1 else 'Normal'
    risk = get_risk_level(float(preds[1]))
    
    print(f'\n--- {label} ---')
    print(f'  Prediction: {pred_cls} (conf: {np.max(preds):.1%})')
    print(f'  P(OSCC):    {preds[1]:.4f}')
    print(f'  Risk Level: {risk}')
    
    # Attention bar chart
    fig, ax = plt.subplots(figsize=(6, 3))
    colors = ['#f87171' if w == attn.max() else '#4ade80' for w in attn]
    ax.bar(range(len(attn)), attn, color=colors, edgecolor='white')
    ax.set_xlabel('Patch Index')
    ax.set_ylabel('Attention Weight')
    ax.set_title(f'{label} — Attention Weights', fontweight='bold')
    ax.set_xticks(range(len(attn)))
    plt.tight_layout()
    plt.show()

## 6. Grad-CAM Explanation
Visualise which regions of the tissue the model focuses on.

In [ ]:
from gradcam import AttentionGradCAM

ag = AttentionGradCAM(model, pe)

for label, img_path in [('Normal', normal_path), ('OSCC', oscc_path)]:
    if img_path is None:
        continue
    save_path = str(PROJECT_ROOT / 'outputs' / 'heatmaps' / f'demo_{label.lower()}.png')
    ag.visualize_prediction(img_path, save_path, true_label=label.lower())
    
    # Display inline
    explanation = plt.imread(save_path)
    fig, ax = plt.subplots(figsize=(18, 5))
    ax.imshow(explanation)
    ax.set_title(f'{label} — Grad-CAM + Attention Explanation', fontsize=14, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.show()

print('\n🔍 Heatmap interpretation:')
print('  🔴 Red regions = high model attention (potential cancer indicators)')
print('  🔵 Blue regions = low attention (likely normal tissue)')
print('  The attention map shows which patches the model considers most important.')

## 7. Evaluation Summary
Load saved metrics and display results.

In [ ]:
import pandas as pd

metrics_path = PROJECT_ROOT / 'outputs' / 'metrics.csv'

if metrics_path.exists():
    df = pd.read_csv(metrics_path)
    print('📊 Evaluation Metrics')
    display(df.T.rename(columns={0: 'Value'}))
else:
    print('⚠️  No metrics file found. Run evaluate.py first.')
    print('    python src/evaluate.py')

# Display saved plots if available
for plot_name, title in [('confusion_matrix.png', 'Confusion Matrix'),
                          ('roc_curve.png', 'ROC Curve')]:
    plot_path = PROJECT_ROOT / 'outputs' / 'plots' / plot_name
    if plot_path.exists():
        img = plt.imread(str(plot_path))
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.imshow(img)
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f'⚠️  {plot_name} not found.')

print('\n✅ Demo complete!')